In [ ]:
import pandas as pd

from parquet_files import COMPOSITE_DATA

composite_data = pd.read_parquet(COMPOSITE_DATA)

In [12]:
composite_data['r1'] = (composite_data['p+1'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r2'] = (composite_data['p+2'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r3'] = (composite_data['p+3'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r4'] = (composite_data['p+4'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r5'] = (composite_data['p+5'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r6'] = (composite_data['p+6'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r7'] = (composite_data['p+7'] - composite_data['p+0']) / composite_data['p+0']

In [13]:
r1_data = composite_data.dropna(subset=['r1']).sort_values('date').reset_index(drop=True)
r2_data = composite_data.dropna(subset=['r2']).sort_values('date').reset_index(drop=True)
r3_data = composite_data.dropna(subset=['r3']).sort_values('date').reset_index(drop=True)
r4_data = composite_data.dropna(subset=['r4']).sort_values('date').reset_index(drop=True)
r5_data = composite_data.dropna(subset=['r5']).sort_values('date').reset_index(drop=True)
r6_data = composite_data.dropna(subset=['r6']).sort_values('date').reset_index(drop=True)
r7_data = composite_data.dropna(subset=['r7']).sort_values('date').reset_index(drop=True)

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error

input_features = ['custom_sentiment', 'finbert_sentiment']

best_rmse = np.inf
best_model = None
best_config = None
perfs = []

for df, target in [(r1_data, 'r1'), (r2_data, 'r2'), (r3_data, 'r3'), (r4_data, 'r4'), (r5_data, 'r5'), (r6_data, 'r6'), (r7_data, 'r7')]:
  for ignore_settings in [[], ['on_volatile_date'], ['on_volatile_date', 'adjacent_volatile_date'], ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date'], ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date', 'near_volatile_date']]:
    for to_ignore in ignore_settings:
      df = df[df[to_ignore] == False].reset_index(drop=True)
    if len(df) < 500:
      continue
    df = df.sort_values('date')
    split_idx = int(len(df) * 0.8)
    train = df.iloc[:split_idx]
    val = df.iloc[split_idx:]
    X_train = train[input_features]
    X_val = val[input_features]
    Y_train = train[target]
    Y_val = val[target]
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, Y_train)
    pred_ridge = ridge.predict(X_val)
    rmse_ridge = root_mean_squared_error(Y_val, pred_ridge)
    perfs.append(("Ridge", ignore_settings, target, rmse_ridge))

    if rmse_ridge < best_rmse:
      best_rmse = rmse_ridge
      best_model = ridge
      best_config = ("Ridge", ignore_settings, target)

    xgb = XGBRegressor(
      n_estimators=200,
      max_depth=3,
      learning_rate=0.05,
      subsample=0.8,
      objective='reg:squarederror',
      tree_method='hist'
    )
    xgb.fit(X_train, Y_train)
    pred_xgb = xgb.predict(X_val)
    rmse_xgb = root_mean_squared_error(Y_val, pred_xgb)
    perfs.append(("XGBoost", ignore_settings, target, rmse_xgb))

    if rmse_xgb < best_rmse:
      best_rmse = rmse_xgb
      best_model = xgb
      best_config = ("XGBoost", ignore_settings, target)

In [15]:
print(perfs)
print(best_rmse)
print(best_config)

[('Ridge', [], 'r1', 0.1243315444296504), ('XGBoost', [], 'r1', 0.12665366394022573), ('Ridge', ['on_volatile_date'], 'r1', 0.12208237770037944), ('XGBoost', ['on_volatile_date'], 'r1', 0.12346461741595484), ('Ridge', ['on_volatile_date', 'adjacent_volatile_date'], 'r1', 0.12301574610558022), ('XGBoost', ['on_volatile_date', 'adjacent_volatile_date'], 'r1', 0.12412481038038296), ('Ridge', ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date'], 'r1', 0.12839562206530386), ('XGBoost', ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date'], 'r1', 0.1286913657676618), ('Ridge', ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date', 'near_volatile_date'], 'r1', 0.15374055180867632), ('XGBoost', ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date', 'near_volatile_date'], 'r1', 0.15558838971221675), ('Ridge', [], 'r2', 0.11650344685374811), ('XGBoost', [], 'r2', 0.1201805503132489), ('Ridge', ['on_volatile_date'], 'r

In [18]:

df = r2_data.copy()
df[df['on_volatile_date'] == False].reset_index(drop=True)
df[df['adjacent_volatile_date'] == False].reset_index(drop=True)

df = df.sort_values('date').reset_index(drop=True)

print(len(df))

split_idx = int(len(df) * 0.8)
test_df = df.iloc[split_idx:]

X_test = test_df[['custom_sentiment', 'finbert_sentiment']]
Y_test = test_df['r2']

39124


In [ ]:
y_pred = best_model.predict(X_test)

sign_match = np.sign(y_pred) == np.sign(Y_test)
sign_accuracy = sign_match.mean()

print(f"Sign accuracy for best model: {sign_accuracy:.3f}")

Sign accuracy for best model: 0.498
